### Zadanie 3 - BONUS - pgvector (symulacja w numpy)

Stworz tabele z 5 opisami filmow i ich embeddingami (mogą być losowe `VECTOR(3)` dla uproszczenia). Napisz zapytanie, ktore znajdzie 3 najbardziej podobne do podanego wektora zapytania.

Wzorujemy sie na DEMO embeddingow z notebooka glownego - wyszukiwanie wektorowe w czystym Pythonie przez `cosine_similarity`.

In [1]:
import numpy as np

# "Baza" filmow z embeddingami (w prawdziwym systemie: OpenAI API + pgvector)
# Wymiary tutaj symboliczne: [akcja/sci-fi, animacja/familyjny, wojenny/dramat]
filmy = {
    "Incepcja":         np.array([0.8, 0.3, 0.9]),
    "Matrix":           np.array([0.75, 0.35, 0.85]),
    "Toy Story":        np.array([0.2, 0.9, 0.1]),
    "Shrek":            np.array([0.25, 0.85, 0.15]),
    "Szeregowiec Ryan": np.array([0.6, 0.1, 0.7]),
}


In [2]:
def cosine_similarity(a, b):
    """Cosine similarity -- 1.0 = identyczne, 0.0 = zupelnie rozne."""
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


def semantic_search(query_vec, database, top_k=3):
    """Zwraca top_k najblizszych filmow wg cosine similarity."""
    results = []
    for title, vec in database.items():
        sim = cosine_similarity(query_vec, vec)
        results.append((title, sim))
    return sorted(results, key=lambda x: -x[1])[:top_k]


In [3]:
# Zapytanie 1 - cos jak sci-fi/akcja
query_scifi = np.array([0.7, 0.3, 0.8])
print(f"Query: {query_scifi}  (sci-fi/akcja)")
for title, sim in semantic_search(query_scifi, filmy, top_k=3):
    bar = "#" * int(sim * 30)
    print(f"  {title:20s} sim={sim:.3f}  {bar}")


Query: [0.7 0.3 0.8]  (sci-fi/akcja)
  Matrix               sim=1.000  #############################
  Incepcja             sim=0.999  #############################
  Szeregowiec Ryan     sim=0.986  #############################


In [4]:
# Zapytanie 2 - cos jak animacja/rodzinny
query_animation = np.array([0.2, 0.9, 0.1])
print(f"Query: {query_animation}  (animacja)")
for title, sim in semantic_search(query_animation, filmy, top_k=3):
    bar = "#" * int(sim * 30)
    print(f"  {title:20s} sim={sim:.3f}  {bar}")


Query: [0.2 0.9 0.1]  (animacja)
  Toy Story            sim=1.000  #############################
  Shrek                sim=0.996  #############################
  Matrix               sim=0.500  ##############


### Rownowaznik w pgvector (PostgreSQL)

Gdybysmy mieli prawdziwy PostgreSQL z rozszerzeniem `vector`, kod wygladalby tak:

```sql
-- Setup
CREATE EXTENSION vector;
CREATE TABLE movies (
    id SERIAL,
    title TEXT,
    embedding VECTOR(3)
);
CREATE INDEX ON movies USING hnsw (embedding vector_cosine_ops);

-- Insert
INSERT INTO movies (title, embedding) VALUES
    ('Incepcja',         '[0.8, 0.3, 0.9]'),
    ('Matrix',           '[0.75, 0.35, 0.85]'),
    ('Toy Story',        '[0.2, 0.9, 0.1]');

-- Top 3 najblizsze (cosine distance)
SELECT title, embedding <=> '[0.7, 0.3, 0.8]'::vector AS distance
FROM movies
ORDER BY embedding <=> '[0.7, 0.3, 0.8]'::vector
LIMIT 3;
```

Operatory: `<->` = L2, `<=>` = cosine, `<#>` = inner product.